# Operations: Data Drift Detection

This notebook mirrors the **Data Drift & Model Decay** guide at [learnmlops.ops4life.com/guides/data-drift/](https://learnmlops.ops4life.com/guides/data-drift/).

Detect when the distribution of incoming data has shifted away from the training distribution using the Kolmogorov-Smirnov test for numeric features and chi-squared for categoricals, with PSI as a severity measure.

**What we'll cover:**
1. What is data drift
2. Generate reference and current datasets
3. Define drift detection functions
4. Detect drift
5. Interpret the results

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from typing import Dict

## Step 1: What is data drift

**Data drift** occurs when the statistical properties of model inputs in production differ from those seen during training. It causes predictions to degrade silently — the model keeps returning outputs, but accuracy deteriorates without any code change.

Two common causes:
- **Covariate shift** — input feature distributions change (e.g., a salary band shifts due to inflation)
- **Concept drift** — the relationship between features and the target changes

We detect covariate drift using:
- **KS test** — non-parametric test comparing two continuous distributions; p < 0.05 indicates drift
- **PSI** — measures magnitude: PSI < 0.1 (stable), 0.1–0.2 (monitor), > 0.2 (retrain)
- **Chi-squared test** — compares categorical frequency distributions

## Step 2: Generate reference and current datasets

In [ ]:
# Reference = training distribution (np.random.seed(42), n=1000)
np.random.seed(42)
reference = pd.DataFrame({
    'MonthlyIncome':     np.random.randint(1500, 20000, 1000),
    'YearsAtCompany':    np.random.randint(0, 20, 1000),
    'TotalWorkingYears': np.random.randint(0, 30, 1000),
    'Age':               np.random.randint(22, 60, 1000),
    'Department':        np.random.choice(['Sales', 'R&D', 'HR'], 1000, p=[0.30, 0.60, 0.10]),
    'OverTime':          np.random.choice(['Yes', 'No'], 1000, p=[0.28, 0.72]),
})

# Current = production distribution with injected drift
np.random.seed(99)
current = pd.DataFrame({
    # Salary shifted up (inflation/market adjustment)
    'MonthlyIncome':     np.random.randint(5000, 25000, 470),
    # Younger workforce (new hiring cohort)
    'Age':               np.random.randint(22, 40, 470),
    # These remain stable
    'YearsAtCompany':    np.random.randint(0, 20, 470),
    'TotalWorkingYears': np.random.randint(0, 30, 470),
    # Department mix shifted
    'Department':        np.random.choice(['Sales', 'R&D', 'HR'], 470, p=[0.45, 0.52, 0.03]),
    'OverTime':          np.random.choice(['Yes', 'No'], 470, p=[0.28, 0.72]),
})

print(f'Reference: {reference.shape}  |  Current: {current.shape}')
print()
print('MonthlyIncome mean — Reference:', reference['MonthlyIncome'].mean().round(),
      '  Current:', current['MonthlyIncome'].mean().round())
print('Age mean — Reference:', reference['Age'].mean().round(1),
      '  Current:', current['Age'].mean().round(1))

The current dataset has intentional drift in two features: `MonthlyIncome` (salary increased) and `Age` (workforce became younger). `YearsAtCompany`, `TotalWorkingYears`, and `OverTime` are stable. The detection functions should surface only the drifted features.

## Step 3: Define drift detection functions

In [ ]:
def calculate_psi(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index — values >0.2 indicate significant drift."""
    ref_hist, bin_edges = np.histogram(reference, bins=bins)
    cur_hist, _         = np.histogram(current, bins=bin_edges)

    ref_pct = ref_hist / len(reference) + 1e-10
    cur_pct = cur_hist / len(current)   + 1e-10

    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return psi


def detect_feature_drift(
    reference: pd.DataFrame,
    current: pd.DataFrame,
    numeric_features: list,
    categorical_features: list,
    significance_level: float = 0.05,
) -> Dict[str, dict]:
    """Detect drift in feature distributions using statistical tests."""
    results = {}

    for feature in numeric_features:
        ref_vals = reference[feature].dropna().values
        cur_vals = current[feature].dropna().values

        stat, p_value = stats.ks_2samp(ref_vals, cur_vals)
        psi = calculate_psi(ref_vals, cur_vals)

        results[feature] = {
            'test':         'ks',
            'statistic':    round(stat, 4),
            'p_value':      round(p_value, 4),
            'drifted':      p_value < significance_level,
            'psi':          round(psi, 4),
            'psi_severity': 'none' if psi < 0.1 else 'minor' if psi < 0.2 else 'major',
        }

    for feature in categorical_features:
        ref_counts = reference[feature].value_counts(normalize=True)
        cur_counts = current[feature].value_counts(normalize=True)

        all_cats = set(ref_counts.index) | set(cur_counts.index)
        ref_freq = np.array([ref_counts.get(c, 0) for c in all_cats])
        cur_freq = np.array([cur_counts.get(c, 0) for c in all_cats])

        n_current = len(current)
        expected  = np.where(ref_freq * n_current < 1, 1, ref_freq * n_current)
        observed  = cur_freq * n_current

        stat, p_value = stats.chisquare(observed, expected)
        results[feature] = {
            'test':      'chi2',
            'statistic': round(stat, 4),
            'p_value':   round(p_value, 4),
            'drifted':   p_value < significance_level,
        }

    return results

## Step 4: Detect drift

In [ ]:
numeric_features     = ['MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 'Age']
categorical_features = ['Department', 'OverTime']

drift_results = detect_feature_drift(
    reference, current,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

## Step 5: Interpret the results

In [ ]:
print(f'{"Feature":<25} {"Test":>5} {"Statistic":>10} {"p-value":>8} {"Drifted":>8} {"PSI":>7} {"Severity":>8}')
print('-' * 80)
for feature, result in drift_results.items():
    psi      = result.get('psi', '-')
    severity = result.get('psi_severity', '-')
    drifted  = '*** YES' if result['drifted'] else 'no'
    psi_str  = f'{psi:.4f}' if isinstance(psi, float) else psi
    print(f'{feature:<25} {result["test"]:>5} {result["statistic"]:>10.4f} '
          f'{result["p_value"]:>8.4f} {drifted:>8} {psi_str:>7} {severity:>8}')

**Interpreting the table:**
- **`drifted=YES`** — p-value is below 0.05; the distributions are statistically distinguishable
- **PSI severity** — quantifies how large the shift is: `none` (<0.1), `minor` (0.1–0.2), `major` (>0.2)
- Features marked `major` should trigger a retraining pipeline immediately
- Features marked `minor` should be monitored over the next few evaluation windows

## Next Steps

- Generate an Evidently HTML report: `04-operations/data-drift-model-decay/evidently_report.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/data-drift/](https://learnmlops.ops4life.com/guides/data-drift/)